<a href="https://colab.research.google.com/github/huanghongjie1024/pytorch_deeplearn/blob/main/_downloads/day3/my_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [2]:
device=torch.accelerator.current_accelerator().type() if torch.accelerator.is_available() else "cpu"
print(f"device is {device}")

device is cpu


In [3]:
training_data=datasets.FashionMNIST(root="data",train=True,download=True,transform=v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32,scale=True)]))
testing_data=datasets.FashionMNIST(root="data",train=False,download=True,transform=v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32,scale=True)]))


100%|██████████| 26.4M/26.4M [00:01<00:00, 19.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 332kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.04MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 19.7MB/s]


In [4]:
train_dataloader=DataLoader(training_data,batch_size=64,shuffle=True)
test_dataloader=DataLoader(testing_data,batch_size=64,shuffle=True)

In [5]:
class NeuralNetwork(nn.Module):
  def __init__(self)->None:
    super().__init__()
    self.flatten=nn.Flatten()
    self.net=nn.Sequential(nn.Linear(28*28,1024),nn.GELU(),nn.Linear(1024,1024),nn.GELU(),nn.Linear(1024,10))

  def forward(self,x):
    x=self.flatten(x)
    logits=self.net(x)
    return logits
model=NeuralNetwork()

In [6]:
learning_rate=1e-3
batch_size=64
epochs=5

In [7]:
loss_fn=nn.CrossEntropyLoss()
opt=torch.optim.Adam(model.parameters(),lr=learning_rate)


In [13]:
def train_loop(dataloader,model,loss_fn,opt):
  size=len(dataloader.dataset)
  model.train()
  for batch,(X,Y) in enumerate(dataloader):
    pred=model(X)
    loss=loss_fn(pred,Y)
    opt.zero_grad();loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
    opt.step()
    if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader,model,loss_fn):
  model.eval()
  num_batch=len(dataloader)
  size=len(dataloader.dataset)
  loss,correct=0.0,0.0
  with torch.no_grad():
    for x,y in dataloader:
      pred=model(x)
      loss+=loss_fn(pred,y).item()
      correct+=(pred.argmax(1)==y).type(torch.float).sum().item()
  loss /= num_batch
  correct /= size
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {loss:>8f} \n")

In [14]:
for i in range(epochs):
  print(f"Epoch {i+1}\n-------------------------------")
  train_loop(train_dataloader,model,loss_fn,opt)
  test_loop(test_dataloader,model,loss_fn)
print("Done")

Epoch 1
-------------------------------
loss: 0.289853  [   64/60000]
loss: 0.307396  [ 6464/60000]
loss: 0.299079  [12864/60000]
loss: 0.429364  [19264/60000]
loss: 0.328681  [25664/60000]
loss: 0.306121  [32064/60000]
loss: 0.255330  [38464/60000]
loss: 0.383783  [44864/60000]
loss: 0.352981  [51264/60000]
loss: 0.262342  [57664/60000]
Test Error: 
 Accuracy: 87.3%, Avg loss: 0.347402 

Epoch 2
-------------------------------
loss: 0.236139  [   64/60000]
loss: 0.323994  [ 6464/60000]
loss: 0.370420  [12864/60000]
loss: 0.145606  [19264/60000]
loss: 0.375666  [25664/60000]
loss: 0.217126  [32064/60000]
loss: 0.304581  [38464/60000]
loss: 0.155965  [44864/60000]
loss: 0.311885  [51264/60000]
loss: 0.217768  [57664/60000]
Test Error: 
 Accuracy: 87.5%, Avg loss: 0.335250 

Epoch 3
-------------------------------
loss: 0.163841  [   64/60000]
loss: 0.236952  [ 6464/60000]
loss: 0.246436  [12864/60000]
loss: 0.267808  [19264/60000]
loss: 0.269448  [25664/60000]
loss: 0.458644  [32064/600